# 估值带 vs 股价：纠偏原则检验

**要回答的问题**（用户 2026-08-14 提出）：

> 对于正确合理的估值方法，股价应当围绕估值上下波动。我认可股价可能存在长时间过分乐观或前瞻化的情况，
> 但在五年/十年的长周期下，估值理应贴近股价变化的中枢。**如果发现一个公司的股价始终高于、
> 或者始终低于算出来的估值，那就说明估值方法存在问题。**

这是个可证伪的判据，且它检验的正是 §9.3 的命门——买入排序完全按 `P/V` 升序，
若 `P/V` 的水平被公司固有偏置主导（而非「当前贵/便宜」），那么排序排的就不是便宜程度，
而是「你属于哪个行业」。

**口径**（`docs/000_Ashare_workflow.md` §6.5.2.1，即 §12.39 全部回测读数所用的那一套）：
`--uniform-tier L2 --roe-source onesided_max --roe-lift 2.0`。

## 复权口径（本 notebook 的可比性前提）

- **股价** = `data/raw/ohlcv/<code>.csv` 的**未复权**收盘。
- **估值带** = 建带引擎按 `split_factor` 做过**送转**折算，与股价的送转跳空同向同幅，**这一侧可比**。
- **现金分红未在带上折算**：实测 `split_factor` 只取 1.0/1.1/1.3/1.56/2.0 等送转因子。
  故每个除息日股价下跳 `D` 而带不动，`P/V` 相应下跳 `D/V`。
  **这是一个已知的口径缺口（§11.3 要求「股价怎么变，带就怎么变」，现实现只做了送转那一半）**，
  单次幅度约等于股息率、对本 notebook 的长周期结论不构成方向性影响，但读图时要知道它在那里。
- `P/V` 面板对任何「两侧同乘一个因子」的复权方式都不变，故**第二张图是复权无关的**，
  是本检验的主证据；第一张图只是把它画得直观。

## 重建输入

本 notebook 读 `data/processed/diag_daily_states.csv`（诊断用逐日估值状态，4.8MB，已 gitignore）。
它不在仓库里，跑之前先建——口径与 §9.3.1.2 采纳口径逐字一致：

```bash
python3 scripts/build_historical_valuation_bands.py \
  --codes 600519,000858,000651,601225,300750,002594,600309,600507,000333,603259 \
  --uniform-tier L2 --roe-source onesided_max --roe-lift 2.0 --since 2002-01-01 \
  --out-bands data/processed/diag_bands.csv \
  --out-daily data/processed/diag_daily_states.csv
```

**`--since 2002-01-01` 必须显式给**——缺省是 `2016-03-31`，不给就只有 2016 年后的样本，
而本检验的全部意义在于长周期（v2.89 曾因此废掉一整轮实测，见 §12.38.0）。

想看全面板，把 `DAILY` 改成 `data/processed/a_share_daily_states_adopted.csv`，并按 `pit_attention/panel_moat_bank_v4.csv`（§9.3.1.2 的基准宇宙，211 只）过滤在册期。


In [ ]:
import csv, collections, statistics, math
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import rcParams
from datetime import date

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# 中文字体：macOS 自带 Heiti SC；换机器时改这里
rcParams["font.sans-serif"] = ["Heiti SC", "Arial Unicode MS", "PingFang SC"]
rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110

# 逐日估值状态。缺省用诊断集；换成 a_share_daily_states_adopted.csv 即可看全面板（在册期以 panel_moat_bank_v4.csv 为准）。
DAILY = ROOT / "data/processed/diag_daily_states.csv"

NAMES = {"600519": "贵州茅台", "000858": "五粮液", "000651": "格力电器", "601225": "陕西煤业",
         "300750": "宁德时代", "002594": "比亚迪", "600309": "万华化学", "600507": "方大特钢",
         "000333": "美的集团", "603259": "药明康德"}

def load(path):
    out = collections.defaultdict(list)
    with open(path, encoding="utf-8") as fh:
        for r in csv.DictReader(fh):
            try:
                iv = float(r["intrinsic_value"]); px = float(r["close"])
            except (TypeError, ValueError, KeyError):
                continue
            if iv <= 0:
                continue
            y, m, d = (int(x) for x in r["date"].split("-"))
            out[r["security_code"]].append((date(y, m, d), px, iv,
                                            float(r["band_low"]), float(r["band_high"])))
    for c in out:
        out[c].sort()
    return out

DATA = load(DAILY)
print(f"载入 {len(DATA)} 只｜" + "、".join(f"{NAMES.get(c,c)} {len(v):,} 日" for c, v in
                                          sorted(DATA.items(), key=lambda kv: -len(kv[1]))[:5]))

## 画图函数

- 上图：股价（蓝）＋ 估值中轴（黑）＋ 买入线 `0.9×中轴`（绿）＋ 卖出线 `1.1×中轴`（红）。
  **对数纵轴**——二十年 40 倍的价格区间用线性轴会把前十年压成一条线。
- 下图：`P/V` 本身，横线标 1.0（纠偏原则要求围绕它震荡）以及 §9.3 实际在用的买入线 1.63、减持线 1.10。

In [ ]:
def plot_band(code, ax_pair=None, buy=0.9, sell=1.1, title_extra=""):
    """一只票的股价-估值带对照图 + P/V 面板。返回该票的诊断读数。"""
    rows = DATA.get(code)
    if not rows:
        raise KeyError(f"{code} 不在当前逐日文件里")
    days = [r[0] for r in rows]
    px   = [r[1] for r in rows]
    iv   = [r[2] for r in rows]
    name = NAMES.get(code, code)

    if ax_pair is None:
        fig, ax_pair = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                                    gridspec_kw={"height_ratios": [3, 1]})
    ax, ax2 = ax_pair

    ax.plot(days, px, lw=1.1, color="#1f77b4", label="股价（未复权收盘）", zorder=3)
    ax.plot(days, iv, lw=1.3, color="black", label="估值中轴", zorder=2)
    ax.plot(days, [v * sell for v in iv], lw=0.9, color="#d62728", label=f"卖出线 {sell}×中轴")
    ax.plot(days, [v * buy for v in iv], lw=0.9, color="#2ca02c", label=f"买入线 {buy}×中轴")
    ax.fill_between(days, [v * buy for v in iv], [v * sell for v in iv],
                    color="0.75", alpha=0.35, zorder=1)
    ax.set_yscale("log")
    ax.set_ylabel("价格（元，对数轴）")
    ax.set_title(f"{name}（{code}）股价 vs 估值带{title_extra}", fontsize=13)
    ax.legend(loc="upper left", fontsize=9, ncol=2)
    ax.grid(alpha=0.25, which="both")

    pv = [p / v for p, v in zip(px, iv)]
    ax2.plot(days, pv, lw=1.0, color="#7f4fa0")
    ax2.axhline(1.0, color="black", lw=1.0)
    ax2.axhline(1.63, color="#d62728", lw=0.8, ls="--")   # §9.3 买入线
    ax2.axhline(1.10, color="#2ca02c", lw=0.8, ls=":")    # §9.3 减持线
    ax2.set_yscale("log")
    ax2.set_ylabel("P/V")
    ax2.grid(alpha=0.25, which="both")
    ax2.xaxis.set_major_locator(mdates.YearLocator(2))
    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    s = sorted(pv)
    diag = {"名称": name, "样本日": len(pv), "起": days[0].isoformat(), "止": days[-1].isoformat(),
            "P/V中位": statistics.median(pv),
            "P/V>1占比": sum(1 for v in pv if v > 1) / len(pv),
            "P5": s[len(s) // 20], "P95": s[len(s) * 19 // 20]}
    ax2.text(0.005, 0.06, f"中位 {diag['P/V中位']:.2f}｜>1 占 {diag['P/V>1占比']*100:.0f}%",
             transform=ax2.transAxes, fontsize=9, color="#7f4fa0")
    return diag

## 一、用户点名的对照：贵州茅台 vs 五粮液

用户的第一条质疑——**「ROE 同样是 30 的话，模型对茅台和五粮液同等对待」**。
两张图并排看：若模型能区分护城河，两者的 `P/V` 中枢应当不同且各自围绕自身合理水平；
若不能，则两者会呈现同一种系统性偏离。

In [ ]:
diags = []
for code in ("600519", "000858"):
    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1]})
    diags.append(plot_band(code, axes))
    plt.tight_layout(); plt.show()

## 二、用户点名的第二条：造车 vs 造电池

**比亚迪 vs 宁德时代**——同处上升期、财务数据形态相近，但业务壁垒完全不同。

In [ ]:
for code in ("002594", "300750"):
    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1]})
    diags.append(plot_band(code, axes))
    plt.tight_layout(); plt.show()

## 三、回测里真正的重仓股长什么样

§12.43.6：格力电器占了全部组合年限的 25.7%、陕西煤业 13.6%、方大特钢 8.9%。
**这三只是策略实际押注的对象**，看看模型对它们的判断是什么形态。

In [ ]:
for code in ("000651", "601225", "600507"):
    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1]})
    diags.append(plot_band(code, axes))
    plt.tight_layout(); plt.show()

## 四、纠偏原则的总表

**判据**：若估值合理，`P/V` 中位应接近 1、`P/V>1` 的天数占比应接近 50%。
偏离越远，说明该票的 `P/V` 水平越是由固有偏置决定、而非由「当前贵/便宜」决定。

In [ ]:
for code in DATA:
    if code not in {d0["名称"] for d0 in diags} and NAMES.get(code, code) not in {d0["名称"] for d0 in diags}:
        fig, axes = plt.subplots(2, 1, figsize=(1, 1))   # 只取读数，不展示
        try:
            diags.append(plot_band(code, axes))
        finally:
            plt.close(fig)

rows = sorted(diags, key=lambda d: -d["P/V中位"])
print(f"{'公司':<9}{'样本日':>7}{'区间':>18}{'P/V中位':>9}{'P/V>1占比':>10}{'P5':>7}{'P95':>7}{'判定':>10}")
for d in rows:
    verdict = "✅ 围绕 1" if 0.8 <= d["P/V中位"] <= 1.25 and 0.35 <= d["P/V>1占比"] <= 0.65 else "❌ 系统性偏离"
    print(f"{d['名称']:<9}{d['样本日']:>7}{d['起'][:7]+'~'+d['止'][:7]:>18}"
          f"{d['P/V中位']:>9.2f}{d['P/V>1占比']*100:>9.0f}%{d['P5']:>7.2f}{d['P95']:>7.2f}{verdict:>12}")

## 五、把偏离画成一张图

横轴 `P/V` 中位、纵轴 `P/V>1` 的天数占比。**合格的估值模型应当把所有点聚在 (1.0, 50%) 附近**。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for d in diags:
    ax.scatter(d["P/V中位"], d["P/V>1占比"] * 100, s=60, color="#1f77b4", zorder=3)
    ax.annotate(d["名称"], (d["P/V中位"], d["P/V>1占比"] * 100),
                textcoords="offset points", xytext=(7, -3), fontsize=10)
ax.axvline(1.0, color="black", lw=1.0)
ax.axhline(50, color="black", lw=1.0)
ax.add_patch(plt.Rectangle((0.8, 35), 0.45, 30, color="#2ca02c", alpha=0.12, zorder=0))
ax.text(1.02, 37, "纠偏原则合格区", color="#2ca02c", fontsize=10)
ax.set_xscale("log")
ax.set_xlabel("P/V 中位（对数轴）"); ax.set_ylabel("P/V > 1 的天数占比（%）")
ax.set_title("纠偏原则检验：每只票的 P/V 长期中枢", fontsize=13)
ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## 结论写在这里

跑完上面的图之后，把观察写进 `docs/Ashare_backtest_log.md`，**不要只留在 notebook 里**——
notebook 是一次性的探查工具，结论必须落到有版本记录的文档中。